In [ ]:
import os
import tensorflow as tf
import hls4ml
import numpy as np

# =====================================================================
# 1. BLOQUE DE PARAMETRIZACIÓN GLOBAL (Link con Etapa 4)
# =====================================================================
# A. Parámetros de Entrenamiento (Origen)
METODO_COMPRESION = "Destillation"
EPOCHS_TRAIN      = 15
BATCH_SIZE_TRAIN  = 128
NOMBRE_MODELO     = "modelo_estudiante_destilado_qat"
BITS_TOTALES      = '8'
BITS_PARTE_ENTERA = '2'

# B. Rutas Dinámicas
DIR_MODELOS   = f"C:/Users/julia/OneDrive/Desktop/IDS-IOT/models/{METODO_COMPRESION}_{EPOCHS_TRAIN}epochs_{BATCH_SIZE_TRAIN}batch"
RUTA_MODELO   = f"{DIR_MODELOS}/{NOMBRE_MODELO}.h5"
DIR_HLS_OUT   = f"hlsPrj/pynq_{NOMBRE_MODELO}_prj"

# C. Parámetros de Arquitectura de Hardware (PYNQ-Z2)
TARGET_BOARD  = 'pynq-z2'
TARGET_PART   = 'xc7z020clg400-1'
IO_PROTOCOL   = 'io_stream' # Requisito para AXI-DMA

# D. Parámetros de Síntesis Matemática (Optimizables según recursos)
PRECISION_HW  = f'ap_fixed<{BITS_TOTALES},{BITS_PARTE_ENTERA}>'
FACTOR_REUSO  = 64                # 1 = Totalmente paralelo (Baja latencia). Subir a 2 o 4 si faltan DSPs.
ESTRATEGIA    = 'Resource'        # Priorizar velocidad ('Latency') vs área ('Resource')

print(f"--- Iniciando Pipeline HLS ---")
print(f"Cargando modelo desde: {RUTA_MODELO}")
print(f"Directorio de salida : {DIR_HLS_OUT}")

--- Iniciando Pipeline HLS ---
Cargando modelo desde: C:/Users/julia/OneDrive/Desktop/IDS-IOT/models/Destillation_15epochs_128batch/modelo_estudiante_destilado_qat.h5
Directorio de salida : hlsPrj/pynq_modelo_estudiante_destilado_qat_prj


In [6]:
# =====================================================================
# 2. CARGA DEL MODELO DESTILADO (Corregido para QKeras)
# =====================================================================
from qkeras.utils import load_qmodel

print(f"Cargando modelo cuantizado desde: {RUTA_MODELO}...")

# Keras puro fallaría aquí. Usamos load_qmodel de QKeras que ya conoce 
# toda la topología de hardware (QDense, QActivation, etc.)
modelo_keras = load_qmodel(RUTA_MODELO, compile=False)

modelo_keras.summary()

Cargando modelo cuantizado desde: C:/Users/julia/OneDrive/Desktop/IDS-IOT/models/Destillation_15epochs_128batch/modelo_estudiante_destilado_qat.h5...
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 qdense_1 (QDense)           (None, 32)                608       
                                                                 
 batch_normalization (Batch  (None, 32)                128       
 Normalization)                                                  
                                                                 
 q_activation (QActivation)  (None, 32)                0         
                                                                 
 qdense_2 (QDense)           (None, 16)                528       
                                                                 
 batch_normalization_1 (Bat  (None, 16)                64        
 chNormalization)                     

In [7]:
# =====================================================================
# 3. CONFIGURACIÓN DEL PERFILADOR HLS4ML
# =====================================================================
config_hls = hls4ml.utils.config_from_keras_model(modelo_keras, granularity='model')

# Aplicamos los parámetros dinámicos de hardware
config_hls['Model']['Precision'] = PRECISION_HW
config_hls['Model']['ReuseFactor'] = FACTOR_REUSO
config_hls['Model']['Strategy'] = ESTRATEGIA

print("\n--- Configuración Matemática Generada ---")
print(config_hls)



--- Configuración Matemática Generada ---
{'Model': {'Precision': 'ap_fixed<8,2>', 'ReuseFactor': 1, 'Strategy': 'Latency', 'BramFactor': 1000000000, 'TraceOutput': False}}


c:\Users\julia\anaconda3\envs\ids-iot-dev\Lib\site-packages\keras\src\constraints.py:365: UserWarning: The `keras.constraints.serialize()` API should only be used for objects of type `keras.constraints.Constraint`. Found an instance of type <class 'qkeras.quantizers.quantized_bits'>, which may lead to improper serialization.
  warnings.warn(


In [9]:
# =====================================================================
# 4. CONVERSIÓN Y COMPILACIÓN (C-SIMULATION)
# =====================================================================
import shutil # Agrega esto junto a tus otros imports arriba (os, tf, hls4ml)

# =====================================================================
# 4. CONVERSIÓN Y COMPILACIÓN (C-SIMULATION)
# =====================================================================
print("\n[1/3] Preparando entorno y convirtiendo modelo Keras a C++...")

if os.path.exists(DIR_HLS_OUT):
    print(f"      -> Detectada compilación previa. Limpiando directorio: {DIR_HLS_OUT}")
    shutil.rmtree(DIR_HLS_OUT)
# ---------------------------------------------------------------------

hls_model = hls4ml.converters.convert_from_keras_model(
    modelo_keras,
    hls_config=config_hls,
    output_dir=DIR_HLS_OUT,
    part=TARGET_PART,
    board=TARGET_BOARD,
    backend='VivadoAccelerator',
    io_type=IO_PROTOCOL
)

print("[2/3] Compilando el modelo en C++...")
hls_model.compile()

# Opcional: Aquí podrías hacer predicciones de prueba con datos
# y_hls = hls_model.predict(X_test_param)


[1/3] Preparando entorno y convirtiendo modelo Keras a C++...
      -> Detectada compilación previa. Limpiando directorio: hlsPrj/pynq_modelo_estudiante_destilado_qat_prj
[2/3] Compilando el modelo en C++...


FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'hlsPrj/pynq_modelo_estudiante_destilado_qat_prj/myproject_test_wrapper.cpp' -> 'hlsPrj/pynq_modelo_estudiante_destilado_qat_prj/myproject_test.cpp'

In [11]:
# =====================================================================
# 5. SÍNTESIS DE HARDWARE Y GENERACIÓN DE BITSTREAM
# =====================================================================
# ATENCIÓN: Esta celda ejecutará Vivado en segundo plano. Puede demorar varios minutos.
print("[3/3] Iniciando Síntesis HLS y Generación de Bitfile...")

# csim=False (ya lo probamos de ser necesario), synth=True (Genera RTL), export=True (Crea IP), bitfile=True (Arma el SoC y compila)
compilation = hls_model.build(csim=False, synth=True, vsynth=True, export=True, bitfile=True)

if compilation:
    print(f"\n--- ¡Proceso Completado! ---")
    print(f"El Bitstream (.bit) y el Handoff (.hwh) se encuentran dentro de: {DIR_HLS_OUT}/vivado_accelerator_project/")
else:
    print("Error")

[3/3] Iniciando Síntesis HLS y Generación de Bitfile...
Project myproject_prj does not exist. Rerun "hls4ml build -p hlsPrj/pynq_modelo_estudiante_destilado_qat_prj".
Project myproject_prj does not exist. Rerun "hls4ml build -p hlsPrj/pynq_modelo_estudiante_destilado_qat_prj".
Error
